In [10]:
import geopandas as gpd
import argparse
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import geopandas as gpd
import networkx as nx
import pandas as pd

# /opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python - <<'PY'
import sys
from pathlib import Path
repo = Path("/Users/6256481/Code/river-hierarchy")
sys.path.insert(0, str(repo))
from hierarchy_level_definition.graph_building.directed_network_checks import analyze_reviewed_network


In [5]:
p = '/Users/6256481/Code/river-hierarchy/network_variants/output/05_indep/states/S063_unit_40/variant/directed/'
links = gpd.read_file(p+'05__S063_unit_40_directed_links.gpkg')
nodes = gpd.read_file(p + '05__S063_unit_40_directed_nodes.gpkg')

In [7]:
from hierarchy_level_definition.graph_building.directed_network_checks import (
    build_directed_graph,
    build_degree_frame,
    validate_single_inlet_single_outlet,
)

graph = build_directed_graph(links, nodes, prefer_explicit_direction=True)
degree_frame = build_degree_frame(graph)
report = validate_single_inlet_single_outlet(graph, require_flag_match=True)

print("is_valid:", report.is_valid)
print("issues:", report.issues)
print("source_nodes:", report.source_nodes)
print("sink_nodes:", report.sink_nodes)
print("flagged_inlets:", report.flagged_inlets)
print("flagged_outlets:", report.flagged_outlets)


is_valid: False
issues: ['Expected exactly one inlet/source node with in_degree == 0 and out_degree > 0, found 2: [80, 154].', 'Expected exactly one outlet/sink node with out_degree == 0 and in_degree > 0, found 2: [0, 82].', 'Expected exactly one node flagged as inlet, found 2: [80, 154].', 'Expected exactly one node flagged as outlet, found 2: [0, 82].']
source_nodes: [80, 154]
sink_nodes: [0, 82]
flagged_inlets: [80, 154]
flagged_outlets: [0, 82]


In [9]:
import networkx as nx

In [11]:
def parse_id_nodes(value: Any) -> tuple[int, int]:
    if value is None or pd.isna(value):
        raise ValueError("Encountered missing id_nodes value.")
    parts = [part.strip() for part in str(value).replace("[", "").replace("]", "").split(",") if part.strip()]
    if len(parts) != 2:
        raise ValueError(f"Expected exactly two node ids in id_nodes, got {value!r}.")
    return int(parts[0]), int(parts[1])

def edge_endpoints_from_row(row: pd.Series, *, prefer_explicit_direction: bool = True) -> tuple[int, int]:
    if prefer_explicit_direction and {"id_us_node", "id_ds_node"}.issubset(row.index):
        us = row.get("id_us_node")
        ds = row.get("id_ds_node")
        if us is not None and ds is not None and not pd.isna(us) and not pd.isna(ds):
            return int(us), int(ds)
    return parse_id_nodes(row["id_nodes"])

In [17]:
graph = nx.MultiDiGraph()

for row in nodes.itertuples(index=False):
    attrs = row._asdict()
    node_id = int(attrs["id_node"])
    graph.add_node(node_id, **attrs)

for _, row in links.iterrows():
    attrs = row.to_dict()
    link_id = int(attrs["id_link"])

    upstream, downstream = edge_endpoints_from_row(row, prefer_explicit_direction=True)
    graph.add_edge(upstream, downstream, key=link_id, **attrs)

    if upstream == 54:
        print('up', link_id)
    if downstream == 54:
        print('dn', link_id)
    # if link_id == 80:
    #     print(upstream)
    #     print(downstream)

    if upstream not in graph:
        graph.add_node(upstream, id_node=upstream)
    if downstream not in graph:
        graph.add_node(downstream, id_node=downstream)

dn 81
up 79
up 80


In [18]:
bad_nodes = sorted(set(report.source_nodes + report.sink_nodes + report.flagged_inlets + report.flagged_outlets))

degree_frame.loc[
    degree_frame["node_id"].isin(bad_nodes),
    ["node_id", "in_degree", "out_degree", "is_source", "is_sink", "flag_is_inlet", "flag_is_outlet"]
].sort_values("node_id")


,node_id,in_degree,out_degree,is_source,is_sink,flag_is_inlet,flag_is_outlet
0,0,1,0,False,True,False,True
80,80,0,3,True,False,True,False
82,82,3,0,False,True,False,True
154,154,0,1,True,False,True,False
